<a href="https://colab.research.google.com/github/OmarAyman2005/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarAyman2005/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
from google.colab import userdata
from huggingface_hub import HfApi
import duckdb
import pandas as pd
import numpy as np
import os

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)
info = api.dataset_info("FlyRank/internship-warehouse")

print("Dataset:", info.id)
print("HF_TOKEN loaded:", HF_TOKEN is not None)
print("Access successful!")

Dataset: FlyRank/internship-warehouse
HF_TOKEN loaded: True
Access successful!


In [8]:
con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

MARCH_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

print("Warehouse connection ready.")
print("Development month: 2026-03")

Warehouse connection ready.
Development month: 2026-03


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1 — CTR vs position

I want to check whether click-through rate differs meaningfully across search-position buckets. This is directly related to FlyRank’s CTR-fix logic. If pages in better positions generally receive higher CTR, then position provides important context when interpreting CTR rather than treating all low-CTR pages equally.

In [9]:
signal1 = con.sql(f"""
WITH page_level AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN 1.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
        END AS ctr,
        AVG(
            CASE
                WHEN gsc_avg_position > 0
                THEN gsc_avg_position
            END
        ) AS avg_position
    FROM read_parquet('{MARCH_PATH}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 100
)

SELECT
    CASE
        WHEN avg_position <= 3 THEN '01: positions 1-3'
        WHEN avg_position <= 10 THEN '02: positions 4-10'
        WHEN avg_position <= 20 THEN '03: positions 11-20'
        ELSE '04: positions 21+'
    END AS position_bucket,
    COUNT(*) AS n,
    ROUND(AVG(ctr), 4) AS mean_ctr
FROM page_level
WHERE avg_position IS NOT NULL
GROUP BY 1
ORDER BY 1
""").df()

signal1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_bucket,n,mean_ctr
0,01: positions 1-3,8295,0.0037
1,02: positions 4-10,46531,0.0032
2,03: positions 11-20,21738,0.0024
3,04: positions 21+,24877,0.0012


Verdict: CONFIRMED. Mean CTR decreases consistently as average position worsens, from 0.0037 for positions 1–3 to 0.0012 for positions 21+. This supports using position context when deciding whether a page’s CTR is weak rather than judging CTR alone.

Signal 2 — Search volume / impressions

I want to check whether pages with more observed search impressions are more likely to deserve priority in a refresh queue. This is related to the quick-win idea: if two pages show similar problems, the one with more search visibility may represent a larger opportunity.

In [10]:
signal2 = con.sql(f"""
WITH page_level AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks
    FROM read_parquet('{MARCH_PATH}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

bucketed AS (
    SELECT
        *,
        CASE
            WHEN impressions < 100 THEN '01: <100'
            WHEN impressions < 500 THEN '02: 100-499'
            WHEN impressions < 2000 THEN '03: 500-1999'
            ELSE '04: 2000+'
        END AS impression_bucket
    FROM page_level
)

SELECT
    impression_bucket,
    COUNT(*) AS n,
    ROUND(AVG(clicks), 2) AS mean_clicks,
    ROUND(AVG(impressions), 2) AS mean_impressions
FROM bucketed
GROUP BY 1
ORDER BY 1
""").df()

signal2

,impression_bucket,n,mean_clicks,mean_impressions
0,01: <100,75297,0.09,24.72
1,02: 100-499,39517,0.57,250.02
2,03: 500-1999,32047,2.88,1054.73
3,04: 2000+,29877,23.45,7869.43


Verdict: CONFIRMED. Pages with larger impression volume also show much higher observed click volume. The 2000+ impression bucket averages 23.45 clicks compared with 0.09 for pages below 100 impressions. This supports using visibility as part of a review-priority rule, because higher-visibility pages may represent a larger opportunity when other signals also suggest underperformance.

Baseline rule: prioritize pages that have meaningful search visibility but relatively weak CTR for their position. The rule gives more weight to pages with higher impressions and lower observed CTR, while using average position as context.

Reason code: visible_low_ctr

Action label: review_for_refresh

This is a transparent hand-written baseline, not a fitted model. It uses only information observed within the development window and no future-window or label-derived inputs.

## 2. Build the ranked queue (writes the CSV)

*I convert the confirmed signals into one transparent baseline score. Each page receives an expected CTR based on its average-position bucket. A page scores higher when it has meaningful search visibility and its observed CTR falls further below the typical CTR for pages in a similar position range.*

*The score is intentionally hand-written and uses no fitted weights. Higher scores mean “higher priority for manual refresh review.”*

In [11]:
import os
import numpy as np
import pandas as pd

# Build one March-level row per content page.
queue = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN 1.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
    END AS ctr,

    AVG(
        CASE
            WHEN gsc_avg_position > 0
            THEN gsc_avg_position
        END
    ) AS avg_position

FROM read_parquet('{MARCH_PATH}')
WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id

HAVING SUM(gsc_impressions) >= 100
""").df()

# Expected CTR from our observed position-bucket audit.
def expected_ctr(position):
    if pd.isna(position):
        return np.nan
    if position <= 3:
        return 0.0037
    elif position <= 10:
        return 0.0032
    elif position <= 20:
        return 0.0024
    else:
        return 0.0012

queue["expected_ctr"] = queue["avg_position"].apply(expected_ctr)

# Relative CTR shortfall.
queue["ctr_shortfall"] = (
    (queue["expected_ctr"] - queue["ctr"]) /
    queue["expected_ctr"]
).clip(lower=0)

# Transparent baseline:
# visibility × relative underperformance.
queue["baseline_score"] = (
    np.log1p(queue["impressions"]) *
    queue["ctr_shortfall"]
)

queue["reason_code"] = "visible_low_ctr"
queue["action_label"] = "review_for_refresh"

queue = queue.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# Write the required CSV.
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "baseline_score",
        "impressions",
        "clicks",
        "ctr",
        "avg_position",
        "expected_ctr",
        "ctr_shortfall",
        "reason_code",
        "action_label"
    ]
].to_csv(output_path, index=False)

print("Ranked pages:", len(queue))
print("CSV written:", output_path)

queue[
    [
        "rank",
        "content_hash_id",
        "baseline_score",
        "impressions",
        "ctr",
        "avg_position",
        "expected_ctr",
        "reason_code",
        "action_label"
    ]
].head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Ranked pages: 101441
CSV written: work/outputs/baseline_action_score.csv


,rank,content_hash_id,baseline_score,impressions,ctr,avg_position,expected_ctr,reason_code,action_label
0,1,content_44f34c0a90047651,11.833128,212404.0,0.000113,7.346909,0.0032,visible_low_ctr,review_for_refresh
1,2,content_8e1334d6356668e3,11.785571,134984.0,0.000007,4.545582,0.0032,visible_low_ctr,review_for_refresh
2,3,content_fec55986a1868d62,11.699109,124075.0,0.000008,9.385150,0.0032,visible_low_ctr,review_for_refresh
3,4,content_559cdd76da9306de,11.289772,97378.0,0.000021,36.712074,0.0012,visible_low_ctr,review_for_refresh
4,5,content_9c057b66c30a3abb,11.280261,83834.0,0.000012,11.967474,0.0024,visible_low_ctr,review_for_refresh
5,6,content_cd3d932d4e1c8db0,11.240607,89332.0,0.000045,7.786219,0.0032,visible_low_ctr,review_for_refresh
6,7,content_164c1f53f13bcee1,11.196086,89982.0,0.000022,24.083947,0.0012,visible_low_ctr,review_for_refresh
7,8,content_046fc480045b88f5,11.082380,83788.0,0.000072,7.289152,0.0032,visible_low_ctr,review_for_refresh
8,9,content_f6116743b00afc2d,11.081226,107584.0,0.000139,9.536301,0.0032,visible_low_ctr,review_for_refresh
9,10,content_425715547c6a3ea8,11.031115,71513.0,0.000042,6.395691,0.0032,visible_low_ctr,review_for_refresh


## 3. Top-10 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top-10 manual review

content_44f34c0a90047651 — Action: review for refresh. Why: extremely high visibility (212,404 impressions) but CTR is far below the expected level for its average position (7.35). Could be wrong if: the page serves an informational result where users often get the answer directly without clicking.
content_8e1334d6356668e3 — Action: review for refresh. Why: 134,984 impressions with almost no clicks despite an average position around 4.55. Could be wrong if: impressions come from queries with naturally very low click intent.
content_fec55986a1868d62 — Action: review for refresh. Why: 124,075 impressions and extremely low CTR while appearing within the first page on average. Could be wrong if: the page appears for many broad or weakly relevant queries.
content_559cdd76da9306de — Action: review for refresh. Why: substantial visibility (97,378 impressions) combined with very low CTR. Could be wrong if: the main problem is poor ranking position (~36.7) rather than the content itself.
content_9c057b66c30a3abb — Action: review for refresh. Why: 83,834 impressions with CTR well below the expected rate for an average position around 12. Could be wrong if: the search-result presentation or query mix explains the low CTR better than page quality.
content_cd3d932d4e1c8db0 — Action: review for refresh. Why: 89,332 impressions and weak CTR despite an average position around 7.79. Could be wrong if: users are satisfied directly on the search-results page or the queries have low click intent.
content_164c1f53f13bcee1 — Action: review for refresh. Why: nearly 90,000 impressions and very low CTR. Could be wrong if: its average position (24.1) is the dominant reason for low CTR rather than a refresh issue.
content_046fc480045b88f5 — Action: review for refresh. Why: 83,788 impressions with CTR substantially below the expected level for an average position around 7.29. Could be wrong if: the query mix creates impressions without strong click intent.
content_f6116743b00afc2d — Action: review for refresh. Why: very high visibility (107,584 impressions) but CTR remains weak for an average position around 9.54. Could be wrong if: the snippet or SERP layout, rather than the underlying content, causes the low click rate.
content_425715547c6a3ea8 — Action: review for refresh. Why: 71,513 impressions with very weak CTR despite an average position around 6.40. Could be wrong if: the queries generating impressions do not align closely with the page's intended audience.

Overall, the ranking surfaces pages with substantial visibility and unusually weak CTR relative to position. These are review candidates, not proof that refreshing the page will improve performance.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Weak picks + leakage check*

*Some high-ranked pages may be weak recommendations even though the baseline score is high. For example, pages with poor average positions may have low CTR mainly because they rank too low, not because the content itself needs refreshing. Similarly, low click intent or SERP features could depress CTR without implying a content problem.*

*The baseline uses only March 2026 observed search signals: impressions, clicks, CTR, and average position. It does not use any future-window outcome, label-derived field, or product flag as an input. The reason code and action label are created by the rule itself after scoring and are not used to calculate the score.*

*Therefore, this queue should be treated as a transparent prioritization baseline for human review, not as proof that the ranked pages need refreshing.*

In [13]:
leakage_check = {
    "uses_future_window": False,
    "uses_label_derived_input": False,
    "uses_product_flag_as_input": False,
    "score_inputs": [
        "impressions",
        "ctr",
        "avg_position"
    ]
}

print("Leakage check:")
for key, value in leakage_check.items():
    print(f"{key}: {value}")

weak_pick = queue.iloc[3]

print("\nExample weak-pick candidate:")
print("Rank:", int(weak_pick["rank"]))
print("Average position:", round(weak_pick["avg_position"], 2))
print(
    "Why potentially weak: low CTR may be explained by poor ranking position rather than refresh need."
)

Leakage check:
uses_future_window: False
uses_label_derived_input: False
uses_product_flag_as_input: False
score_inputs: ['impressions', 'ctr', 'avg_position']

Example weak-pick candidate:
Rank: 4
Average position: 36.71
Why potentially weak: low CTR may be explained by poor ranking position rather than refresh need.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.